# 1. Data Cleaning
**Dataset:** Customer Churn (Telecom) — `data/raw_data.csv`
**Goal:** Load raw data, assess quality, clean, and export `data/cleaned_data.csv` for EDA.


In [4]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/raw_data.csv')
df.shape

(500, 9)

## 1.1 First Look

In [5]:
df.head()

,CustomerID,Tenure,MonthlyCharges,TotalCharges,Contract,PaymentMethod,PaperlessBilling,SeniorCitizen,Churn
0,C00001,6,64,1540,One year,Credit Card,No,1,0
1,C00002,21,113,1753,Month-to-month,Electronic Check,Yes,1,0
2,C00003,27,31,1455,Two year,Credit Card,No,1,0
3,C00004,53,29,7150,Month-to-month,Electronic Check,No,1,0
4,C00005,16,185,1023,One year,Electronic Check,No,1,0


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   CustomerID        500 non-null    object
 1   Tenure            500 non-null    int64 
 2   MonthlyCharges    500 non-null    int64 
 3   TotalCharges      500 non-null    int64 
 4   Contract          500 non-null    object
 5   PaymentMethod     500 non-null    object
 6   PaperlessBilling  500 non-null    object
 7   SeniorCitizen     500 non-null    int64 
 8   Churn             500 non-null    int64 
dtypes: int64(5), object(4)
memory usage: 35.3+ KB


In [7]:
df.describe(include='all')

,CustomerID,Tenure,MonthlyCharges,TotalCharges,Contract,PaymentMethod,PaperlessBilling,SeniorCitizen,Churn
count,500,500.000000,500.000000,500.000000,500,500,500,500.000000,500.000000
unique,500,NaN,NaN,NaN,3,3,2,NaN,NaN
top,C00500,NaN,NaN,NaN,One year,Credit Card,No,NaN,NaN
freq,1,NaN,NaN,NaN,186,178,257,NaN,NaN
mean,NaN,36.532000,113.636000,4237.882000,NaN,NaN,NaN,0.498000,0.106000
std,NaN,20.667057,51.799903,2260.619837,NaN,NaN,NaN,0.500497,0.308146
min,NaN,1.000000,20.000000,159.000000,NaN,NaN,NaN,0.000000,0.000000
25%,NaN,19.000000,67.000000,2237.250000,NaN,NaN,NaN,0.000000,0.000000
50%,NaN,37.000000,115.000000,4182.500000,NaN,NaN,NaN,0.000000,0.000000
75%,NaN,54.000000,158.000000,6266.750000,NaN,NaN,NaN,1.000000,0.000000


## 1.2 Data Quality Checks

In [8]:
# Missing values
print("Missing values per column:\n", df.isnull().sum())

Missing values per column:
 CustomerID          0
Tenure              0
MonthlyCharges      0
TotalCharges        0
Contract            0
PaymentMethod       0
PaperlessBilling    0
SeniorCitizen       0
Churn               0
dtype: int64


In [9]:
# Duplicate rows / duplicate IDs
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate CustomerIDs:", df['CustomerID'].duplicated().sum())

Duplicate rows: 0
Duplicate CustomerIDs: 0


In [10]:
# Data types
df.dtypes

CustomerID          object
Tenure               int64
MonthlyCharges       int64
TotalCharges         int64
Contract            object
PaymentMethod       object
PaperlessBilling    object
SeniorCitizen        int64
Churn                int64
dtype: object

In [11]:
# Range / sanity checks on numeric columns
for col in ['Tenure', 'MonthlyCharges', 'TotalCharges']:
    print(col, '-> min:', df[col].min(), 'max:', df[col].max())

Tenure -> min: 1 max: 71
MonthlyCharges -> min: 20 max: 199
TotalCharges -> min: 159 max: 7992


In [12]:
# Category value checks
for col in ['Contract', 'PaymentMethod', 'PaperlessBilling', 'SeniorCitizen', 'Churn']:
    print(col, ':', df[col].unique())

Contract : ['One year' 'Month-to-month' 'Two year']
PaymentMethod : ['Credit Card' 'Electronic Check' 'Bank Transfer']
PaperlessBilling : ['No' 'Yes']
SeniorCitizen : [1 0]
Churn : [0 1]


**Quality summary (documented for reproducibility):**
- No missing values found.
- No duplicate rows or CustomerIDs.
- Numeric columns (Tenure, MonthlyCharges, TotalCharges) are within sane ranges — no negatives.
- Categorical columns have clean, consistent labels — no typos or inconsistent casing detected.


## 1.3 Cleaning Steps

In [13]:
df_clean = df.copy()

# Drop exact duplicate rows (safety net, even if none found above)
df_clean = df_clean.drop_duplicates()

# Standardize column dtypes
df_clean['SeniorCitizen'] = df_clean['SeniorCitizen'].astype(int)
df_clean['Churn'] = df_clean['Churn'].astype(int)

# Encode binary Yes/No as 1/0 for downstream modeling convenience
df_clean['PaperlessBilling_Flag'] = df_clean['PaperlessBilling'].map({'Yes': 1, 'No': 0})

# Strip whitespace from string/object columns
obj_cols = df_clean.select_dtypes(include='object').columns
df_clean[obj_cols] = df_clean[obj_cols].apply(lambda s: s.str.strip())

df_clean.shape

(500, 10)

In [14]:
# Feature engineering: useful derived column for later analysis
df_clean['AvgMonthlySpend'] = (df_clean['TotalCharges'] / df_clean['Tenure'].replace(0, 1)).round(2)
df_clean.head()

,CustomerID,Tenure,MonthlyCharges,TotalCharges,Contract,PaymentMethod,PaperlessBilling,SeniorCitizen,Churn,PaperlessBilling_Flag,AvgMonthlySpend
0,C00001,6,64,1540,One year,Credit Card,No,1,0,0,256.67
1,C00002,21,113,1753,Month-to-month,Electronic Check,Yes,1,0,1,83.48
2,C00003,27,31,1455,Two year,Credit Card,No,1,0,0,53.89
3,C00004,53,29,7150,Month-to-month,Electronic Check,No,1,0,0,134.91
4,C00005,16,185,1023,One year,Electronic Check,No,1,0,0,63.94


## 1.4 Final Validation

In [15]:
assert df_clean.isnull().sum().sum() == 0, "Nulls remain!"
assert df_clean.duplicated().sum() == 0, "Duplicates remain!"
print("Validation passed. Final shape:", df_clean.shape)

Validation passed. Final shape: (500, 11)


## 1.5 Export Cleaned Data

In [16]:
df_clean.to_csv('../data/cleaned_data.csv', index=False)
print("Saved: data/cleaned_data.csv")

Saved: data/cleaned_data.csv


In [17]:
print("hellow world")

hellow world
